# Semantic roundtrip: thesis figures

Set the six completed study-job paths and the completed `STYLE_REPORT_DIR` below,
restart the kernel and **Run All**. Figures appear inline and are saved as PNG/PDF.
Supporting tables and provenance are exported quietly to `OUTPUT_DIR`.

RQ1–RQ3 concern model configurations and reconstruction routes. SQ1–SQ5 cover
prompt reconstruction, visual style, native thinking, title domains and illustratability.
`style_decision.ipynb` is the sole complete four-style analysis. This notebook validates
and imports its frozen compact summary, including photorealistic results, without
recalculating the style comparison. The 900-candidate report remains separate.

Use the exact unrestricted Direct source for SQ1, Thinking and the frozen style report.
Image/description routes have four planned observations per title and condition;
the prompt route has two. Missing planned outcomes remain zero.


In [ ]:
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.ticker import MaxNLocator

from semantic_roundtrip.analysis import aggregate_titles, load_job
from semantic_roundtrip.analysis.plotting import (
    bb_heatmaps,
    heatmap,
    interval_plot,
    rating_analysis,
    save_figure,
)
from semantic_roundtrip.analysis.reporting import (
    DOMAINS,
    METRIC,
    PAIRS,
    QG,
    TEXT,
    annotate,
    difference,
    effects,
    export_tables,
    indirect_contrasts,
    load_direct_supplement,
    matching_contrasts,
    overall_domain_means,
    technical_tables,
    write_manifest,
)

# Silence only the known pandas deprecation; data/errors are not suppressed.
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    message="The behavior of DataFrame concatenation with empty or all-NA entries is deprecated.*",
)

# Run from the repository root or its notebooks/ directory.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sns.set_theme(
    style="whitegrid",
    context="notebook",
    rc={"pdf.fonttype": 42, "ps.fonttype": 42, "axes.unicode_minus": False},
)
from semantic_roundtrip.analysis.prompt_baseline import (
    load_prompt_baseline,
    plot_prompt_baseline,
    prompt_baseline_tables,
)
from semantic_roundtrip.analysis.style_report import (
    display_style_summary,
    load_style_summary,
)


In [ ]:
# Replace the absolute example paths, or set the corresponding environment variables.
# A job path is its directory, not an individual child run or a website URL.
DIRECT_JOB = os.getenv("DIRECT_JOB", "/absolute/path/to/direct-job")
INDIRECT_JOB = os.getenv("INDIRECT_JOB", "/absolute/path/to/local-indirect-job")
AQUEDUCT_JOB = os.getenv("AQUEDUCT_JOB", "/absolute/path/to/matching-aqueduct-job")
THINKING_JOB = os.getenv("THINKING_JOB", "/absolute/path/to/matching-thinking-job")
ILLUSTRATABLE_JOB = os.getenv(
    "ILLUSTRATABLE_JOB", "/absolute/path/to/illustratable-job"
)
PROMPT_BASELINE_JOB = os.getenv("PROMPT_BASELINE_JOB", "/absolute/path/to/final-direct-prompt-only")
STYLE_REPORT_DIR = os.getenv("STYLE_REPORT_DIR", "/absolute/path/to/completed-style-report")
OUTPUT_DIR = (
    Path(os.getenv("OUTPUT_DIR", ROOT / "notebooks/results/final"))
    .expanduser()
    .resolve()
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Reading the figures

Primary accuracy requires a passed prompt check, a passed strict image-text check
and Strict Exact Match. Rejected, failed and missing observations score zero in
the full planned denominator. Four seed observations
are averaged per title. Intervals use 10,000 paired whole-title bootstrap samples,
stratified only by domain (seed 20260829); 95% intervals are pointwise.
Accuracy is in percent; differences are in percentage points (pp).
Supporting exports include prediction-only accuracy, normalized title matching
and the separate title-aware image policy. Undefined estimates remain `n/a`.

## A — RQ1: Direct model configurations

### Load once

Run this cell for the 4×4 analysis and the paired-route comparison. No indirect jobs are needed.

In [ ]:
direct = load_job(DIRECT_JOB)
direct_obs = annotate(direct.observations)
direct_titles = aggregate_titles(direct_obs, condition_columns=["pg", "bb", "bi"])
direct_only = direct_titles[direct_titles.route == "direct"]

### Family, generation and PG/BI combination

**Question:** How do model family, model generation, and the combination of prompt-generation
and image-interpretation models relate to direct end-to-end title-reconstruction accuracy
for the selected Qwen and Gemma models?

Start with the complete 4×4 heatmap, then examine the four planned 2×2 comparisons.
For cells A=XX, B=XY, C=YX, D=YY:
PG = (C+D−A−B)/2; BI = (B+D−A−C)/2; same−mixed = (A+D−B−C)/2.
The last is a PG×BI interaction contrast, not an independent additional mechanism.
BI comparisons reuse images; PG comparisons legitimately use different generated images.

In [ ]:
direct_cells = 100 * direct_only.groupby(["pg", "bi"])[METRIC].mean().unstack().reindex(
    index=QG, columns=QG
)
fig, ax = plt.subplots(figsize=(6, 5), layout="constrained")
fig.colorbar(
    heatmap(ax, direct_only, QG, ""), ax=ax, label="End-to-end Strict Exact Match (%)"
)
save_figure(
    fig, OUTPUT_DIR / "direct_4x4", "RQ1: Direct reconstruction across PG and BI models"
)

In [ ]:
direct_contrasts = {}
for label, (x, y) in PAIRS.items():
    xx, xy, yx, yy = [
        f"direct_pg_{pg}_bi_{bi}" for pg, bi in [(x, x), (x, y), (y, x), (y, y)]
    ]
    pair_label = f"{label} ({y.upper()} − {x.upper()})"
    direct_contrasts[f"PG: {pair_label}"] = difference([yx, yy], [xx, xy])
    direct_contrasts[f"BI: {pair_label}"] = difference([xy, yy], [xx, yx])
    direct_contrasts[f"Same − mixed: {label}"] = difference([xx, yy], [xy, yx])
direct_effects = effects(direct_only, direct_contrasts)
roles = ["PG", "BI", "Same − mixed"]
panel_titles = [
    "Prompt generation (PG)",
    "Image interpretation (BI)",
    "Model pairing (same - mixed)",
]
comparison_colors = {
    "Qwen": "#0072B2",
    "Gemma": "#D55E00",
    "Family 2025": "#009E73",
    "Family 2026": "#CC79A7",
}
overall = direct_effects[direct_effects.domain == "all"]
x_min = 5 * np.floor((overall.ci95_low.min() - 3) / 5)
x_max = 5 * np.ceil((overall.ci95_high.max() + 3) / 5)
fig, axes = plt.subplots(1, 3, figsize=(14, 4.8), sharex=True, layout="constrained")
for role, title, ax in zip(roles, panel_titles, axes):
    table = direct_effects[
        (direct_effects.domain == "all")
        & direct_effects.comparison.str.startswith(role + ":")
    ].copy()
    table["comparison"] = table.comparison.str.removeprefix(role + ": ").str.replace(
        "−", "-", regex=False
    )
    interval_plot(ax, table, colors=list(comparison_colors.values()))
    ax.set(title=title, xlim=(x_min, x_max))
fig.supxlabel(
    "Positive PG/BI values favour the first model in each subtraction; positive pairing values favour same-model pairs.",
    fontsize=9,
    color="#444444",
)
save_figure(
    fig,
    OUTPUT_DIR / "direct_primary_effects",
    "RQ1: Direct reconstruction - planned model contrasts",
)

In [ ]:
family = {"q25": "qwen", "q38": "qwen", "g3": "gemma", "g4": "gemma"}
cohort = {"q25": 2025, "g3": 2025, "q38": 2026, "g4": 2026}
relation = np.select(
    [
        direct_only.pg.eq(direct_only.bi),
        direct_only.pg.map(family).eq(direct_only.bi.map(family)),
        direct_only.pg.map(cohort).eq(direct_only.bi.map(cohort)),
    ],
    ["Same model", "Same family, other generation", "Same generation, other family"],
    default="Different family and generation",
)
direct_relations = (
    100 * direct_only.assign(relation=relation).groupby("relation")[METRIC].mean()
).rename("accuracy_percent")

## B — RQ2: Description-mediated model configurations

### Load once

Load the completed local 16-condition job and its complete 20-condition Aqueduct extension.
Together they form the 36-cell matrix. The hosted extension must inherit from this local job.

In [ ]:
local = load_job(INDIRECT_JOB)
aqueduct = load_job(AQUEDUCT_JOB)
local_obs = annotate(local.observations)
aqueduct_obs = annotate(aqueduct.observations)
local_titles = aggregate_titles(local_obs, condition_columns=["pg", "bb", "bi"])
aqueduct_titles = aggregate_titles(aqueduct_obs, condition_columns=["pg", "bb", "bi"])
full_obs = pd.concat([local_obs, aqueduct_obs], ignore_index=True)
full_titles = pd.concat([local_titles, aqueduct_titles], ignore_index=True)
full_ratings = pd.concat([local.ratings, aqueduct.ratings], ignore_index=True)

### PG/BB/BI combinations

**Question:** How do the selection and combination of the prompt-generation, image-description,
and description-interpretation models relate to title-reconstruction accuracy on the
description-mediated route?

Four BB panels show the complete 3×4×3 matrix. The six local effects retain their
predeclared D32/O120 scope. Two separately labelled hosted effects compare V4 with the
mean of D32 and O120 over the full matrix. Matching is supporting, not the definition of RQ2.
These are deployed-configuration comparisons, not isolated architecture or age effects.

In [ ]:
bb_heatmaps(
    full_titles,
    TEXT,
    OUTPUT_DIR / "indirect_3x4x3_V4_hosted",
    "RQ2: Description-mediated reconstruction (V4 externally hosted)",
)

In [ ]:
local_effects = effects(local_titles, indirect_contrasts(local_titles)).assign(
    scope="D32/O120 submatrix"
)
hosted_contrasts = {
    name: weights
    for name, weights in indirect_contrasts(full_titles).items()
    if "V4" in name
}
hosted_effects = effects(full_titles, hosted_contrasts).assign(
    scope="Full 3×4×3 matrix"
)
indirect_effects = pd.concat([local_effects, hosted_effects], ignore_index=True)
plot_effects = indirect_effects[indirect_effects.domain == "all"].copy()
plot_effects["comparison"] = plot_effects.scope + ": " + plot_effects.comparison
fig, ax = plt.subplots(figsize=(9, 5), layout="constrained")
interval_plot(ax, plot_effects)
save_figure(
    fig,
    OUTPUT_DIR / "indirect_primary_effects",
    "RQ2: Description-mediated reconstruction - model-role contrasts",
)

In [ ]:
local_matching = effects(local_titles, matching_contrasts(local_titles)).assign(
    scope="D32/O120 submatrix"
)
full_matching = effects(full_titles, matching_contrasts(full_titles)).assign(
    scope="Full 3×4×3 matrix"
)
indirect_matching = pd.concat([local_matching, full_matching], ignore_index=True)

## C — RQ3: Paired reconstruction routes

**Question:** For the same generated images, how does direct image-to-title reconstruction
differ from reconstruction through an explicit image description in the selected multimodal
baseline conditions?

Only the four diagonal conditions provide this comparison (BB=BI). Show absolute accuracies
next to description−direct differences: four baselines and their equal-weight mean.
A route difference does not by itself prove information loss. Transition counts below
describe images; the confidence intervals still use titles as the statistical unit.

In [ ]:
diagonal = direct_titles[direct_titles.pg == direct_titles.bi]
route_means = (
    100 * diagonal.groupby(["pg", "route"])[METRIC].mean().unstack()
).reindex(QG)
route_scores = diagonal.assign(
    condition_route=diagonal.condition + "__" + diagonal.route
)
route_contrasts = {
    m.upper(): difference(
        [f"direct_pg_{m}_bi_{m}__description"], [f"direct_pg_{m}_bi_{m}__direct"]
    )
    for m in QG
}
route_contrasts["Mean of four baselines"] = difference(
    [f"direct_pg_{m}_bi_{m}__description" for m in QG],
    [f"direct_pg_{m}_bi_{m}__direct" for m in QG],
)
route_effects = effects(
    route_scores, route_contrasts, condition_column="condition_route"
)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
for route, marker, offset in [("direct", "o", -0.08), ("description", "s", 0.08)]:
    axes[0].plot(
        np.arange(4) + offset,
        route_means[route],
        marker=marker,
        linestyle="none",
        label=route,
    )
axes[0].set(
    xticks=range(4),
    xticklabels=[m.upper() for m in QG],
    xlabel="Baseline (PG=BB=BI)",
    ylabel="End-to-end Strict Exact Match (%)",
    ylim=(-2, 102),
)
axes[0].legend()
interval_plot(axes[1], route_effects[route_effects.domain == "all"])
axes[0].set_title("Absolute accuracy")
axes[1].set_title("Description - direct")
save_figure(
    fig,
    OUTPUT_DIR / "paired_routes",
    "RQ3: Direct and description-mediated reconstruction",
)
transitions = (
    direct_obs[direct_obs.pg == direct_obs.bi]
    .pivot(
        index=["condition", "domain", "item_key", "prompt_seed", "image_seed"],
        columns="route",
        values="end_to_end_strict_score",
    )
    .reset_index()
)
transition_order = [
    "Both correct",
    "Direct only",
    "Description only",
    "Neither correct",
]
transitions["outcome"] = np.select(
    [
        transitions.direct.eq(1) & transitions.description.eq(1),
        transitions.direct.eq(1),
        transitions.description.eq(1),
    ],
    transition_order[:3],
    default=transition_order[3],
)
route_transitions = (
    transitions.groupby(["condition", "domain", "outcome"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=transition_order, fill_value=0)
)

## D — Secondary analyses

These analyses preserve the primary random, unrestricted, thinking-off study.
SQ1 adds prompt reconstruction without regenerating the existing image routes.
SQ2 imports the complete four-style report; SQ3 changes the native thinking policy.
SQ4 compares title domains; SQ5 relates model-rated illustratability to reconstruction
and compares the random and high-illustratability datasets descriptively.


### SQ1 — Prompt and image reconstruction

**Question:** How does title reconstruction from the original generated prompt compare
with direct image reconstruction, and with description-mediated reconstruction in the
four existing same-model diagonal baselines?

All sixteen cells reuse the exact unrestricted source prompts, images and image predictions.
Only 2,880 prompt guesses are new (180 per cell). The three-route subset reuses 720 of
those guesses and 1,440 existing observations per image route; it is not a second study.
Direct versus description is the same evidence as RQ3, not an independent replication.

The prompt score requires prompt verification and Strict Exact Match; image routes also
require strict image verification. Missing outcomes are zero. Two prompt observations
and four image observations are averaged within each title before paired, domain-stratified
whole-title bootstrap. Normalized Exact Match is sensitivity. Differences measure route-level
recoverability, not the unique location of semantic loss or a guaranteed prompt upper bound.


In [ ]:
sq1, sq1_obs, sq1_titles = load_prompt_baseline(PROMPT_BASELINE_JOB, DIRECT_JOB)
sq1_tables = prompt_baseline_tables(sq1_obs, sq1_titles)
sq1_tables["sq1_rq3_reference"] = route_effects.assign(
    evidence="Existing RQ3 Direct/Indirect contrast; reused, not independent replication"
)
sq1_tables.update({
    f"sq1_{name}": table
    for name, table in technical_tables(
        {"SQ1": sq1_obs}, {"SQ1": sq1_titles}, [("SQ1", sq1)]
    ).items()
    if name not in {"error_attempts", "task_time_minutes"}
})
plot_prompt_baseline(sq1_titles, sq1_tables, OUTPUT_DIR)


The supporting `sq1_common_valid_inputs.csv` restricts all compared routes to the
same prompt-check-pass and strict-image-check-pass inputs. It reports eligible counts
and descriptive means with missing predictions still zero. This selected-subset diagnostic
does not replace the full-denominator analysis. All sixteen prompt/direct cells and the
four-diagonal three-route subset are reported separately; their pooled means are not mixed.


### SQ2 — Frozen four-style evidence

The following compact panel is imported unchanged from the completed style report.
All four styles share ninety titles and sixteen model cells. Primary and sensitivity
metrics retain their full denominators; pooled intervals retain original domain strata.
The source report contains the detailed paired effects and manipulation diagnostics.

Unrestricted generation is the least-intervention reference recommendation, subject
to supervisor confirmation after the complete report. Descriptive centrality measures
distance from the other styles; it does not automatically select a style or maximise accuracy.


In [ ]:
style_summary, style_provenance = load_style_summary(
    STYLE_REPORT_DIR, DIRECT_JOB, OUTPUT_DIR
)


In [ ]:
display_style_summary(style_provenance["directory"])


In [ ]:
style_centrality = style_summary["style_centrality"]
style_centrality[
    style_centrality.domain.eq("all") & style_centrality.metric.eq(METRIC)
][["style", "centrality_pp", "centrality_rank", "tied", "tie_count"]]


In [ ]:
# Detailed frozen CSVs, source manifest and panel PNG/PDF files are copied to
# OUTPUT_DIR/style_summary; hashes and methods were validated before display.


### SQ3 — Native-thinking deployment comparison

The native-thinking matrix uses the twelve newly calculated cells plus the four unchanged
Q25/G3 cells from the unrestricted Direct job. Absolute matrices and a native-minus-off
difference heatmap are accompanied by paired effects for all 16 cells, the 12 changed cells,
and three disjoint four-cell groups: native thinking in PG only, BI only, or both roles.
The groups contain different PG/BI pairs; their differences do not isolate a role interaction.
This is a deployment-policy comparison, including changed output ceilings and timeouts,
not an isolated causal effect of thinking or equal test-time compute.

In [ ]:
thinking, thinking_obs, thinking_changed = load_direct_supplement(THINKING_JOB)
unchanged = direct_only[
    direct_only.pg.isin(["q25", "g3"]) & direct_only.bi.isin(["q25", "g3"])
]
thinking_direct = pd.concat([unchanged, thinking_changed], ignore_index=True)
if thinking_direct.condition.nunique() != 16:
    raise ValueError(
        "Thinking matrix does not contain 12 changed and 4 baseline cells."
    )
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), layout="constrained")
image = heatmap(axes[0], direct_only, QG, "Thinking off")
heatmap(axes[1], thinking_direct, QG, "Native thinking for Q38/G4")
delta = heatmap(axes[2], thinking_direct, QG, "Native - off", baseline=direct_only)
fig.colorbar(image, ax=list(axes[:2]), label="End-to-end Strict Exact Match (%)")
fig.colorbar(delta, ax=axes[2], label="Difference (pp)")
save_figure(fig, OUTPUT_DIR / "direct_thinking_matrices", "SQ3: Native thinking")

In [ ]:
off_scores = direct_only.assign(
    condition_mode=lambda f: "off__" + f.condition.astype(str)
)
native_scores = thinking_direct.assign(
    condition_mode=lambda f: "native__" + f.condition.astype(str)
)
combined = pd.concat([off_scores, native_scores], ignore_index=True)
cells = thinking_direct[["condition", "pg", "bi"]].drop_duplicates()
native_pg = cells.pg.isin(["q38", "g4"])
native_bi = cells.bi.isin(["q38", "g4"])
thinking_groups = {
    "All 16 cells": cells.condition,
    "12 changed cells": cells.loc[native_pg | native_bi, "condition"],
    "PG only (4 cells)": cells.loc[native_pg & ~native_bi, "condition"],
    "BI only (4 cells)": cells.loc[~native_pg & native_bi, "condition"],
    "PG and BI (4 cells)": cells.loc[native_pg & native_bi, "condition"],
}
thinking_contrasts = {
    f"{label}: native - off": difference(
        native_scores.loc[
            native_scores.condition.isin(conditions), "condition_mode"
        ].unique(),
        off_scores.loc[
            off_scores.condition.isin(conditions), "condition_mode"
        ].unique(),
    )
    for label, conditions in thinking_groups.items()
}
thinking_effects = effects(
    combined, thinking_contrasts, condition_column="condition_mode"
)
plotted = thinking_effects[thinking_effects.domain == "all"].copy()
plotted["comparison"] = plotted.comparison.str.split(":").str[0]
fig, ax = plt.subplots(figsize=(9, 4.5), layout="constrained")
interval_plot(ax, plotted)
save_figure(
    fig, OUTPUT_DIR / "direct_thinking_effects", "SQ3: Native thinking - thinking off"
)

### SQ4 — Domain comparison

Run sections A and B first. This section combines no accuracy across designs; it presents
domain means side by side. Section E consolidates diagnostics after loading supplements.

In [ ]:
direct_domain_effects = direct_effects[direct_effects.domain != "all"].copy()
route_domain_effects = route_effects[route_effects.domain != "all"].copy()

In [ ]:
direct_domains = overall_domain_means(direct_only, "Direct 4×4")
indirect_domains = overall_domain_means(full_titles, "Indirect 3×4×3")
domain_results = pd.concat([direct_domains, indirect_domains], ignore_index=True)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5), layout="constrained")
offsets = {"Direct 4×4": -0.1, "Indirect 3×4×3": 0.1}
markers = {"Direct 4×4": "o", "Indirect 3×4×3": "s"}
for design, table in domain_results.groupby("design", sort=False):
    table = table.set_index("domain").reindex(DOMAINS)
    y = np.arange(len(DOMAINS)) + offsets[design]
    errors = np.vstack(
        [table.estimate - table.ci95_low, table.ci95_high - table.estimate]
    )
    ax.errorbar(
        table.estimate, y, xerr=errors, fmt=markers[design], capsize=2, label=design
    )
ax.set(
    yticks=range(len(DOMAINS)),
    yticklabels=list(DOMAINS),
    xlim=(0, 100),
    xlabel="End-to-end Strict Exact Match (%), pointwise 95% CI",
)
ax.legend()
save_figure(
    fig, OUTPUT_DIR / "domain_overview", "SQ4: Reconstruction accuracy by domain"
)

### SQ5 — Model-rated illustratability

#### Association on the direct route

Correlations use each PG model's rating and accuracy averaged across all four BI models, direct route only.
The three domain histograms give each title one value: the equal mean of its Q25, G3, Q38 and G4 ratings
from the Direct job. All four ratings are required; missing means are reported separately.
Technical validity is exported in section E.

In [ ]:
title_characteristics = direct_obs[
    ["dataset_id", "item_key", "domain", "expected_title", "title_length_group"]
].drop_duplicates()
title_length_counts = (
    title_characteristics.groupby(["domain", "title_length_group"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=DOMAINS, columns=["short", "medium", "long"], fill_value=0)
)
fig, ax = plt.subplots(figsize=(7, 3.5), layout="constrained")
sns.countplot(
    data=title_characteristics,
    x="domain",
    hue="title_length_group",
    hue_order=["short", "medium", "long"],
    ax=ax,
)
ax.set(xlabel="Domain", ylabel="Titles")
save_figure(
    fig, OUTPUT_DIR / "title_length_distribution", "Title-length distribution by domain"
)

In [ ]:
direct_rating_summary, direct_rating_pairs = rating_analysis(
    direct.ratings,
    direct_only,
    OUTPUT_DIR / "direct_illustratability",
    "SQ5: Illustratability and direct reconstruction",
)

In [ ]:
rating_distribution = direct.ratings.reindex(
    columns=["entry_name", "dataset_id", "domain", "item_key", "score"]
).copy()
rating_distribution["pg"] = rating_distribution.entry_name.str.extract(
    "_pg_([^_]+)_", expand=False
)
rating_index = ["dataset_id", "domain", "item_key"]
rating_means = (
    rating_distribution.drop_duplicates([*rating_index, "pg"])
    .pivot(index=rating_index, columns="pg", values="score")
    .reindex(columns=QG)
    .mean(axis=1, skipna=False)
    .rename("mean_score")
)
rating_means = title_characteristics[rating_index].merge(
    rating_means, on=rating_index, how="left"
)
rating_counts = (
    rating_means.groupby("domain")
    .mean_score.agg(titles="size", complete_ratings="count")
    .reindex(DOMAINS)
)
rating_counts["missing_ratings"] = rating_counts.titles - rating_counts.complete_ratings
fig, axes = plt.subplots(
    1, 3, figsize=(12, 3.8), sharex=True, sharey=True, layout="constrained"
)
for domain, ax in zip(DOMAINS, axes):
    sns.histplot(
        data=rating_means[rating_means.domain == domain],
        x="mean_score",
        color=DOMAINS[domain][0],
        bins=np.arange(0, 101, 10),
        ax=ax,
    )
    ax.set(
        title=f"{domain.title()} (n={rating_counts.loc[domain, 'complete_ratings']})",
        xlabel="Mean illustratability (0–100)",
        ylabel="Titles",
        xlim=(0, 100),
    )
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
save_figure(
    fig,
    OUTPUT_DIR / "illustratability_distribution",
    "SQ5: Model-averaged illustratability by domain",
)

#### Association on the description-mediated route

Every PG rating is related to accuracy over the same four BB × three BI downstream set.
Domain results are shown under SQ4; technical validity is exported in section E.

In [ ]:
indirect_domain_effects = indirect_effects[indirect_effects.domain != "all"].copy()
indirect_domain_matching = indirect_matching[indirect_matching.domain != "all"].copy()
full_rating_summary, full_rating_pairs = rating_analysis(
    full_ratings,
    full_titles,
    OUTPUT_DIR / "indirect_illustratability",
    "SQ5: Illustratability and description-mediated reconstruction",
)

#### Random versus high-illustratability titles

The two datasets contain different titles. The matrices and differences below are descriptive
and do not estimate a paired, causal or population-wide effect of filtering.

In [ ]:
illustratable, illustratable_obs, illustratable_direct = load_direct_supplement(
    ILLUSTRATABLE_JOB
)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), layout="constrained")
image = heatmap(axes[0], direct_only, QG, "Random titles")
heatmap(axes[1], illustratable_direct, QG, "High illustratability")
delta = heatmap(
    axes[2], illustratable_direct, QG, "Selected - random", baseline=direct_only
)
fig.colorbar(image, ax=list(axes[:2]), label="End-to-end Strict Exact Match (%)")
fig.colorbar(delta, ax=axes[2], label="Difference (pp)")
save_figure(
    fig,
    OUTPUT_DIR / "direct_illustratability_matrices",
    "SQ5: High-illustratability vs random titles (descriptive)",
)
descriptive_dataset_comparison = pd.DataFrame(
    {
        "dataset": ["Random", "High illustratability"],
        "titles": [
            direct_only.item_key.nunique(),
            illustratable_direct.item_key.nunique(),
        ],
        "mean_accuracy_percent": [
            100 * direct_only[METRIC].mean(),
            100 * illustratable_direct[METRIC].mean(),
        ],
    }
)

## E — Technical validity and supporting accuracy

Run the preceding sections first. Thinking includes the
four unchanged baseline cells in its accuracy, but does not count them as new computation.
Counts represent condition/route observations, not unique images or fresh model calls.
Rows with condition or domain `all` are pooled totals: do not add them to detail rows.
Every table is saved as CSV; undefined values are displayed as `n/a`.
Imported errors and timings are counted only once, at their original local source.
SQ1 contributes only its new prompt route to pooled route diagnostics. The imported
Direct/Indirect observations are displayed in SQ1 but are not new main-study evidence.
Style diagnostics remain available in the validated frozen style report.


In [ ]:
observation_sets = {"Direct": direct_obs, "Indirect 3×4×3": full_obs}
title_sets = {"Direct": direct_titles, "Indirect 3×4×3": full_titles}
job_sets = [("Direct", direct), ("Indirect 3×4×3", local), ("Indirect 3×4×3", aqueduct)]
for label, job, observations, scores in [
    ("Thinking", thinking, thinking_obs, thinking_direct),
    ("Illustratable", illustratable, illustratable_obs, illustratable_direct),
]:
    observations = observations[observations.route.eq("direct")]
    if label == "Thinking":
        baseline = direct_obs[
            direct_obs.route.eq("direct")
            & direct_obs.pg.isin(["q25", "g3"])
            & direct_obs.bi.isin(["q25", "g3"])
        ]
        observations = pd.concat([observations, baseline], ignore_index=True)
    observation_sets[label] = observations
    title_sets[label] = scores
    job_sets.append((label, job))
observation_sets["SQ1 prompt (new)"] = sq1_obs[sq1_obs.route.eq("prompt")]
title_sets["SQ1 prompt (new)"] = sq1_titles[sq1_titles.route.eq("prompt")]
job_sets.append(("SQ1 prompt (new)", sq1))
technical = technical_tables(observation_sets, title_sets, job_sets)
for name, table in technical.items():
    table.to_csv(OUTPUT_DIR / f"{name}.csv", index=False, na_rep="n/a")

## Reproducibility exports

All figure data and technical tables are saved below. Verification counts distinguish
one deterministic prompt decision per prompt from two model decisions per image.
Imported errors and timings are deduplicated by provenance; durations are task
minutes, not an estimate of end-to-end wall time. Hashes document the inputs;
they do not enforce title-set compatibility.

In [ ]:
export_tables(
    {
        "direct_cells": direct_cells.reset_index(),
        "direct_effects": direct_effects,
        "direct_relations": direct_relations.reset_index(),
        "paired_routes": route_effects,
        "route_transitions": route_transitions.reset_index(),
        "title_length_counts": title_length_counts.reset_index(),
        "direct_illustratability": direct_rating_summary,
        "direct_illustratability_pairs": direct_rating_pairs,
        "rating_means": rating_means,
        "rating_counts": rating_counts.reset_index(),
        "indirect_effects": indirect_effects,
        "indirect_matching": indirect_matching,
        "indirect_illustratability": full_rating_summary,
        "indirect_illustratability_pairs": full_rating_pairs,
        "domain_results": domain_results,
        "thinking_effects": thinking_effects,
        "dataset_comparison": descriptive_dataset_comparison,
        **sq1_tables,
    },
    OUTPUT_DIR,
)
write_manifest(
    ROOT / "notebooks/final_study.ipynb",
    {
        "Direct": DIRECT_JOB,
        "Indirect": INDIRECT_JOB,
        "Aqueduct": AQUEDUCT_JOB,
        "Thinking": THINKING_JOB,
        "Illustratable": ILLUSTRATABLE_JOB,
        "Prompt baseline": PROMPT_BASELINE_JOB,
    },
    OUTPUT_DIR,
    analysis={
        "purpose": "full_study",
        "seed_observations_per_title": {"direct": 4, "description": 4, "prompt": 2},
        "style_report": style_provenance,
        "sq1": {"source_style": "Unrestricted", "prompt_direct_cells": 16, "three_route_diagonals": 4, "new_planned_predictions": 2880},
        "primary": METRIC,
        "denominator": "all planned observations",
        "bootstrap": {
            "unit": "paired whole title",
            "strata": ["domain"],
            "repetitions": 10000,
            "seed": 20260829,
            "pointwise": True,
        },
    },
)